# EvalHub - Create Evaluation Job

This notebook demonstrates how to create an evaluation job using the EvalHub REST API.

## Prerequisites
- EvalHub deployed in `redhat-ods-applications` namespace
- Model deployed (e.g., `tinyllama-1b` in `models-as-a-service` namespace)
- Model API token/key available
- User has MLflow RBAC permissions (see RoleBinding in 13-mlflow)

## Running in RHOAI Workbench
This notebook automatically detects when running inside an RHOAI workbench and retrieves:
- OpenShift authentication token from `/var/run/secrets/kubernetes.io/serviceaccount/token`
- Cluster domain from namespace routes
- Current namespace from the environment

In [20]:
import requests
import json
import os
import subprocess
from urllib3.exceptions import InsecureRequestWarning
import warnings

# Suppress SSL warnings for demo purposes (use proper certs in production)
warnings.simplefilter('ignore', InsecureRequestWarning)

## Auto-detect RHOAI Workbench Environment

In [21]:
def get_openshift_token():
    """Get OpenShift token from workbench ServiceAccount or environment."""
    # Try reading from ServiceAccount token (RHOAI workbench)
    token_path = '/var/run/secrets/kubernetes.io/serviceaccount/token'
    if os.path.exists(token_path):
        with open(token_path, 'r') as f:
            print("✓ Running in RHOAI workbench - using ServiceAccount token")
            return f.read().strip()
    
    # Fall back to environment variable
    token = os.getenv('OPENSHIFT_TOKEN')
    if token:
        print("✓ Using token from OPENSHIFT_TOKEN environment variable")
        return token
    
    # Last resort - try oc command
    try:
        result = subprocess.run(['oc', 'whoami', '-t'], capture_output=True, text=True, check=True)
        print("✓ Retrieved token using 'oc whoami -t'")
        return result.stdout.strip()
    except:
        print("⚠ Could not auto-detect token. Please set manually.")
        return 'YOUR_TOKEN_HERE'

def get_cluster_domain():
    """Get OpenShift cluster domain from routes or environment."""
    # Try from environment variable first
    domain = os.getenv('CLUSTER_DOMAIN')
    if domain:
        return domain
    
    # Try to detect from well-known route
    try:
        result = subprocess.run(
            ['oc', 'get', 'route', '-n', 'openshift-console', 'console', '-o', 'jsonpath={.spec.host}'],
            capture_output=True, text=True, check=True
        )
        console_host = result.stdout.strip()
        # Extract domain from console-openshift-console.apps.ocp.sandbox5548.opentlc.com
        if 'apps.' in console_host:
            domain = console_host.split('apps.', 1)[1]
            print(f"✓ Detected cluster domain: {domain}")
            return domain
    except:
        pass
    
    # Default fallback
    print("⚠ Using default cluster domain - update if needed")
    return 'ocp.sandbox5548.opentlc.com'

def get_current_namespace():
    """Get current namespace from workbench or environment."""
    # Try reading from ServiceAccount namespace (RHOAI workbench)
    namespace_path = '/var/run/secrets/kubernetes.io/serviceaccount/namespace'
    if os.path.exists(namespace_path):
        with open(namespace_path, 'r') as f:
            return f.read().strip()
    
    # Try oc command
    try:
        result = subprocess.run(['oc', 'project', '-q'], capture_output=True, text=True, check=True)
        return result.stdout.strip()
    except:
        return None

# Auto-detect configuration
OCP_TOKEN = get_openshift_token()
CLUSTER_DOMAIN = get_cluster_domain()
CURRENT_NAMESPACE = get_current_namespace()

if CURRENT_NAMESPACE:
    print(f"✓ Current namespace: {CURRENT_NAMESPACE}")

✓ Retrieved token using 'oc whoami -t'
✓ Detected cluster domain: ocp.sandbox5548.opentlc.com
✓ Current namespace: rhoai-playground


## Configuration

Set up the EvalHub endpoint and model configuration

In [22]:
# EvalHub API endpoint (auto-detected from cluster domain)
EVALHUB_URL = f"https://evalhub-redhat-ods-applications.apps.{CLUSTER_DOMAIN}"

# Model configuration
# Change MODEL_NAME to match your deployed model
MODEL_NAMESPACE = "models-as-a-service"
MODEL_NAME = "gpt-oss-20b"  # Change to: tinyllama-1b, gpt-oss-20b, etc.
MODEL_URL = f"https://maas.apps.{CLUSTER_DOMAIN}/{MODEL_NAMESPACE}/{MODEL_NAME}"

# Get model API secret name (not the token value!)
# EvalHub expects secret_ref to be a Kubernetes secret NAME that exists
def get_model_secret_name(model_name, namespace):
    """Get model ServiceAccount secret name for EvalHub auth.
    
    Returns the SECRET NAME, not the token value.
    EvalHub will mount this secret and read the token from it.
    """
    # Check if user provided a custom secret name
    secret_name = os.getenv('MODEL_SECRET_NAME')
    if secret_name:
        print(f"✓ Using MODEL_SECRET_NAME from environment: {secret_name}")
        return secret_name
    
    # Default: use the ServiceAccount token secret
    secret_name = f"{model_name}-sa-token"
    
    # Verify the secret exists
    try:
        result = subprocess.run(
            ['oc', 'get', 'secret', secret_name, '-n', namespace],
            capture_output=True, text=True, check=True
        )
        print(f"✓ Using model secret: {secret_name} (verified in namespace {namespace})")
        return secret_name
    except Exception as e:
        print(f"⚠ Secret {secret_name} not found in namespace {namespace}")
        print(f"  Available secrets: oc get secrets -n {namespace} | grep {model_name}")
        return secret_name  # Return anyway, let EvalHub fail with clear error

MODEL_SECRET_NAME = get_model_secret_name(MODEL_NAME, MODEL_NAMESPACE)

# Request headers for EvalHub API
headers = {
    "Authorization": f"Bearer {OCP_TOKEN}",
    "Content-Type": "application/json",
    "X-Tenant": MODEL_NAMESPACE
}

print(f"\n📋 Configuration:")
print(f"  EvalHub URL: {EVALHUB_URL}")
print(f"  Model URL: {MODEL_URL}")
print(f"  Auth token: {OCP_TOKEN[:20]}...{OCP_TOKEN[-10:]}")
print(f"  Model secret: {MODEL_SECRET_NAME}")

✓ Using MODEL_API_TOKEN from environment

📋 Configuration:
  EvalHub URL: https://evalhub-redhat-ods-applications.apps.ocp.sandbox5548.opentlc.com
  Model URL: https://maas.apps.ocp.sandbox5548.opentlc.com/models-as-a-service/tinyllama-1b
  Auth token: sha256~6neQ_DMiUnvtF...AlsOE-JJPM
  Model token: sk-jx3718g6l1yQb2wpS...


## Step 1: Check EvalHub Health

In [23]:
response = requests.get(
    f"{EVALHUB_URL}/api/v1/health",
    headers=headers,
    verify=False
)

print(f"Status Code: {response.status_code}")
print(f"Response: {response.text}")

Status Code: 200
Response: {"status":"healthy","timestamp":"2026-06-15T06:36:40.247774246Z","build":"0.3.0"}



## Step 2: List Available Providers

In [27]:
response = requests.get(
    f"{EVALHUB_URL}/api/v1/evaluations/providers",
    headers=headers,
    verify=False
)

print(f"Status Code: {response.status_code}")
if response.status_code == 200:
    data = response.json()
    providers = data.get("items", [])
    print(f"\nAvailable Providers ({data.get('total_count', 0)}):")
    for provider in providers:
        resource = provider.get("resource", {})
        print(f"  • {provider.get('name', 'Unnamed')} (id: {resource.get('id', 'N/A')})")
        print(f"    {provider.get('description', 'No description').strip()}")
else:
    print(f"Error: {response.text}")

Status Code: 200

Available Providers (3):
  • LM Evaluation Harness (id: lm_evaluation_harness)
    Comprehensive evaluation framework for language models with 180 benchmarks
  • GuideLLM (id: guidellm)
    Performance benchmarking framework for LLM inference servers
  • Garak (id: garak)
    LLM vulnerability scanner and red-teaming framework


## Step 3: List Available Collections

In [26]:
response = requests.get(
    f"{EVALHUB_URL}/api/v1/evaluations/collections",
    headers=headers,
    verify=False
)

print(f"Status Code: {response.status_code}")
if response.status_code == 200:
    data = response.json()
    collections = data.get("items", [])
    print(f"\nAvailable Collections ({data.get('total_count', 0)}):")
    for collection in collections:
        resource = collection.get("resource", {})
        print(f"  • {collection.get('name', 'Unnamed')} (id: {resource.get('id', 'N/A')})")
        print(f"    {collection.get('description', 'No description').strip()}")
        benchmarks = collection.get("benchmarks", [])
        print(f"    Contains {len(benchmarks)} benchmarks")
else:
    print(f"Error: {response.text}")

Status Code: 200

Available Collections (1):
  • Safety & Fairness (id: safety-and-fairness-v1)
    Evaluates model safety, bias, and fairness across diverse scenarios.
    Contains 6 benchmarks


## Step 4: Create Evaluation Job

Create an evaluation job for the Arc Easy benchmark

In [ ]:
# Evaluation job configuration
# ✅ CORRECT: Pass the SECRET NAME (not token value)
evaluation_job = {
    "name": "EvalHub Eval GPT-20B",
    "tenant": MODEL_NAMESPACE,
    "model": {
        "name": MODEL_NAME,
        "url": MODEL_URL,
        "auth": {
            "secret_ref": MODEL_SECRET_NAME  # Secret NAME (e.g., "gpt-oss-20b-sa-token")
        }
    },
    "benchmarks": [
        {
            "id": "arc_easy",  # Arc Easy benchmark
            "provider_id": "lm_evaluation_harness",
            "primary_score": {
                "metric": "acc_norm"
            },
            "pass_criteria": {
                "threshold": 0.25
            },
            "parameters": {
                "limit": 5,  # Limit to 5 samples for quick testing
                "num_fewshot": 0,
                # Specify the HuggingFace tokenizer model ID
                # For GPT models, use the base GPT-2 tokenizer or specific model variant
                # For TinyLlama: "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
                # For GPT models: "openai-community/gpt2" or "EleutherAI/gpt-neo-20b"
                "tokenizer": "EleutherAI/gpt-neo-20b"  # HuggingFace model ID for tokenizer
            }
        }
    ],
    "experiment": {
        "name": "EvalHub GPT test"
    }
}

print("Creating evaluation job with config:")
print(json.dumps({
    **evaluation_job,
    "model": {**evaluation_job["model"], "auth": {"secret_ref": MODEL_SECRET_NAME}}
}, indent=2))
print(f"\n✅ Using secret: {MODEL_SECRET_NAME}")
print("   EvalHub will mount this secret as a volume in the evaluation pod")
print(f"\n💡 Tokenizer: {evaluation_job['benchmarks'][0]['parameters']['tokenizer']}")
print("   Must match the HuggingFace model ID for tokenization")

Creating evaluation job with config:
{
  "name": "EvalHub Eval 01",
  "tenant": "models-as-a-service",
  "model": {
    "name": "tinyllama-1b",
    "url": "https://maas.apps.ocp.sandbox5548.opentlc.com/models-as-a-service/tinyllama-1b",
    "auth": {
      "secret_ref": "<REDACTED>"
    }
  },
  "benchmarks": [
    {
      "id": "arc_easy",
      "provider_id": "lm_evaluation_harness",
      "primary_score": {
        "metric": "acc_norm"
      },
      "pass_criteria": {
        "threshold": 0.25
      },
      "parameters": {
        "limit": 10,
        "num_fewshot": 0
      }
    }
  ],
  "experiment": {
    "name": "EvalHub test 01"
  }
}

⚠ WARNING: Due to RHOAI 3.4.0 bug, the job will be created but will fail in Pending status
  Error: MountVolume.SetUp failed - secret with JWT token as name cannot be created
  See README.md Known Issues section for details


In [25]:
response = requests.post(
    f"{EVALHUB_URL}/api/v1/evaluations/jobs",
    headers=headers,
    json=evaluation_job,
    verify=False
)

print(f"Status Code: {response.status_code}")
print(f"\nResponse:")
try:
    print(json.dumps(response.json(), indent=2))
    
    if response.status_code == 200 or response.status_code == 201:
        job_data = response.json()
        job_id = job_data.get('id', job_data.get('job_id'))
        print(f"\n✓ Evaluation job created successfully!")
        print(f"  Job ID: {job_id}")
        print(f"  Status: {job_data.get('status', 'unknown')}")
        
        # Save job ID for later use
        with open('evalhub_job_id.txt', 'w') as f:
            f.write(job_id)
        print(f"\n  Job ID saved to: evalhub_job_id.txt")
    else:
        print(f"\n✗ Failed to create evaluation job")
except:
    print(f"Response text: {response.text}")
    print(f"\n✗ Failed to create evaluation job")

Status Code: 400

Response:
{
  "message_code": "request_validation_failed",
  "message": "The request validation failed: 'Key: 'EvaluationJobConfig.benchmarks' Error:Field validation for 'benchmarks' failed on the 'minimum one benchmark' tag'. Please check the request and try again.",
  "trace": "63174c62-c7d9-4a03-9289-abad8fc4eb3e"
}

✗ Failed to create evaluation job


## Troubleshooting

If you encounter errors:

### 1. Authentication Issues
```python
# Verify your token is valid
!oc whoami

# Get fresh token if needed
import subprocess
result = subprocess.run(['oc', 'whoami', '-t'], capture_output=True, text=True)
print(f"Token: {result.stdout.strip()[:20]}...")
```

### 2. MLflow RBAC Permissions
```bash
# Check if RoleBinding exists
!oc get rolebinding evalhub-mlflow-user-access -n models-as-a-service

# Check your groups
!oc whoami
!oc get groups rhods-admins -o yaml
```

### 3. Model Token Retrieval
```bash
# Manually get model token
!oc get secret tinyllama-1b-sa-token -n models-as-a-service -o jsonpath='{.data.token}' | base64 -d
```

### 4. Check EvalHub Pod
```bash
!oc get pods -n redhat-ods-applications -l app=evalhub
!oc logs -n redhat-ods-applications deployment/evalhub --tail=50
```

### 5. Known RHOAI 3.4.0 Bug
EvalHub may incorrectly use the API key token as a secret name instead of creating a proper secret reference. This causes jobs to fail with "secret not found" errors. See the README for workarounds.